In [6]:
import os
import rasterio
from importlib_resources import files
from pathlib import Path
from rasterio.enums import Resampling
from tqdm import tqdm

from beak.utilities.raster_processing import unify_raster_grids
from beak.utilities.io import save_raster, create_file_list, check_path, create_file_folder_list


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

In [7]:
BASE_PATH = files("beak.data")

TARGET_RES = 2240
TARGET_EPSG = 102008
SPATIAL_EXTENT = "US_ALASKA"

PATH_ROOT = BASE_PATH / "LAWLEY22" / "EXPORT"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / f"EPSG_{TARGET_EPSG}_RES_{TARGET_RES}_{SPATIAL_EXTENT}.tif"
PATH_INPUT = PATH_ROOT / "EPSG_4326_RES_0_02" / "US_CANADA"

# Export options
PATH_EXPORT = PATH_ROOT / f"EPSG_{TARGET_EPSG}_RES_{TARGET_RES}" / SPATIAL_EXTENT

# Optional example export
CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name

# Set selected output path
OUT_FOLDER = PATH_EXPORT

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\LAWLEY22\EXPORT\EPSG_4326_RES_0_02\US_CANADA
Export folder: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\LAWLEY22\EXPORT\EPSG_102008_RES_2240\US_ALASKA


Select subfolders to be scaled.

In [8]:
folders = os.listdir(PATH_INPUT)
for folder in folders:
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


CATEGORICAL
GROUND_TRUTH
NUMERICAL
NUMERICAL_IMPUTED
NUMERICAL_IMPUTED_SCALED_STANDARD
NUMERICAL_SCALED_STANDARD


**Run unification**

In [9]:
from beak.utilities.io import create_file_folder_list
folders, files = create_file_folder_list(PATH_INPUT)

print(f"Total of folders found: {len(folders)}")
print(f"Total of files found: {len(files)}")

Total of folders found: 52
Total of files found: 955


In [10]:
import multiprocessing as mp

for folder in tqdm(folders):
  folder_relative = os.path.relpath(folder, PATH_INPUT)
  files = create_file_list(folder)
  
  if len(files) > 0:
    result = unify_raster_grids(BASE_RASTER, files, resampling_mode="auto", same_extent=True, same_shape=True, n_workers=mp.cpu_count())
    
    output_folder = OUT_FOLDER / str(folder_relative)
    for i, file in enumerate(files):
      raster = rasterio.open(file)
      out_path = output_folder / file.name
      check_path(Path(os.path.dirname(out_path)))
      save_raster(out_path, array=result[i][0], metadata=result[i][1], dtype=raster.meta["dtype"], overwrite=True)
  

100%|██████████| 52/52 [22:18<00:00, 25.73s/it]
